In [ ]:
!pip install optuna scikit-learn seaborn matplotlib -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import time
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score, 
    f1_score, 
    recall_score, 
    precision_score,
    hamming_loss,
    fbeta_score
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2

import optuna

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
DATASET_PATH = "/kaggle/input/anomaly-windowed-test1/windowed"
if not os.path.exists(DATASET_PATH):
    print("ERROR: No se encuentra el dataset. Verifica la ruta.")
    print("Ruta probada:", DATASET_PATH)
    os.makedirs(DATASET_PATH, exist_ok=True)
else:
    print(f"Dataset encontrado en: {DATASET_PATH}")

In [ ]:
try:
    X = np.load(os.path.join(DATASET_PATH, "X_windows.npy"))
    y_bin = np.load(os.path.join(DATASET_PATH, "y_binary.npy"))
    y_ch = np.load(os.path.join(DATASET_PATH, "y_channel.npy"))
    
    N, W, F = X.shape
    
    print("Dataset cargado exitosamente!")
    print("=" * 50)
    print(f"Total de ventanas: {N}")
    print(f"Tamaño de ventana: {W}")
    print(f"Número de características: {F}")
    print(f"Ventanas anómalas: {sum(y_bin)}")
    print(f"Ventanas normales: {N - sum(y_bin)}")
    print(f"Porcentaje de anomalías: {100 * sum(y_bin)/N:.2f}%")
    print(f"Forma de y_channel: {y_ch.shape}")
    
    # Verificar si y_ch es one-hot encoding
    if len(y_ch.shape) == 2:
        num_classes = y_ch.shape[1]
        print(f"Detectado: y_ch parece ser one-hot encoding con {num_classes} clases")
        # Convertir a etiquetas multiclase si es necesario
        y_multiclass = np.argmax(y_ch, axis=1) if len(y_ch.shape) == 2 else y_ch
        print(f"Clases únicas en y_multiclass: {np.unique(y_multiclass)}")
    
except Exception as e:
    print(f"Error cargando dataset: {e}")
    print("Verifica que los archivos .npy existan en la ruta especificada.")

In [ ]:
idx_nom = np.where(y_bin == 0)[0]
idx_anom = np.where(y_bin == 1)[0]

print(f"Ventanas normales: {len(idx_nom)}")
print(f"Ventanas anómalas: {len(idx_anom)}")

X_nom = X[y_bin == 0]  # Solo datos normales

mean_feat = X_nom.mean(axis=(0, 1))  # (num_features,)
std_feat = X_nom.std(axis=(0, 1)) + 1e-8

def scale_windows(X, mean, std):
    return (X - mean[None, None, :]) / std[None, None, :]

X_scaled = scale_windows(X, mean_feat, std_feat)

print("Preprocesamiento completado!")
print(f"X_scaled shape: {X_scaled.shape}")
print(f"X_scaled - Media: {X_scaled.mean():.4f}, Std: {X_scaled.std():.4f}")

In [ ]:
all_indices = np.arange(len(X_scaled))

train_idx_full, temp_idx_full = train_test_split(
    all_indices,
    test_size=0.3,
    random_state=SEED,
    stratify=y_bin  
)

val_idx_full, test_idx_full = train_test_split(
    temp_idx_full,
    test_size=0.5,
    random_state=SEED,
    stratify=y_bin[temp_idx_full]
)

X_train_full = X_scaled[train_idx_full]
y_bin_train_full = y_bin[train_idx_full]
y_ch_train_full = y_ch[train_idx_full]

X_val_full = X_scaled[val_idx_full]
y_bin_val_full = y_bin[val_idx_full]
y_ch_val_full = y_ch[val_idx_full]

X_test_full = X_scaled[test_idx_full]
y_bin_test_full = y_bin[test_idx_full]
y_ch_test_full = y_ch[test_idx_full]

print(f"\nSplits completos (70/15/15):")
print(f"  Train: {len(X_train_full)} muestras")
print(f"  Val:   {len(X_val_full)} muestras")
print(f"  Test:  {len(X_test_full)} muestras")
print(f"  Distribución train: {np.bincount(y_bin_train_full.astype(int))}")
print(f"  % Anomalías en train: {100*sum(y_bin_train_full)/len(y_bin_train_full):.1f}%")

X_train = X_train_full
y_bin_train = y_bin_train_full
y_ch_train = y_ch_train_full
X_val = X_val_full
y_bin_val = y_bin_val_full
y_ch_val = y_ch_val_full
X_test = X_test_full
y_bin_test = y_bin_test_full
y_ch_test = y_ch_test_full

if len(y_ch.shape) == 2:
    # y_ch ya está en one-hot encoding
    y_multi_train = y_ch_train
    y_multi_val = y_ch_val
    y_multi_test = y_ch_test
    num_classes = y_ch.shape[1]
    print(f"\nPara multiclase: {num_classes} clases detectadas")
else:
    # Convertir a one-hot si no lo está
    num_classes = len(np.unique(y_ch))
    y_multi_train = tf.keras.utils.to_categorical(y_ch_train, num_classes=num_classes)
    y_multi_val = tf.keras.utils.to_categorical(y_ch_val, num_classes=num_classes)
    y_multi_test = tf.keras.utils.to_categorical(y_ch_test, num_classes=num_classes)
    print(f"\nConvertido a one-hot: {num_classes} clases")

print("DATA FOR TRAINNING")
print(f"X_train shape: {X_train.shape}")
print(f"y_bin_train distribución: {np.bincount(y_bin_train.astype(int))}")
print(f"y_multi_train shape: {y_multi_train.shape if 'y_multi_train' in locals() else 'N/A'}")

In [ ]:
def create_lstm_binary(input_shape, lstm_units=64, dropout_rate=0.2, learning_rate=0.001):
    model = models.Sequential([
        layers.LSTM(lstm_units, return_sequences=True, input_shape=input_shape,
                   kernel_regularizer=l2(0.001)),
        layers.Dropout(dropout_rate),
        layers.LSTM(lstm_units//2, return_sequences=False),
        layers.Dropout(dropout_rate),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy', 
                tf.keras.metrics.Precision(name='precision'),
                tf.keras.metrics.Recall(name='recall'),
                tf.keras.metrics.AUC(name='auc')]
    )
    
    return model

def create_lstm_multiclass(input_shape, num_classes, lstm_units=128, dropout_rate=0.3, learning_rate=0.001):
    model = models.Sequential([
        layers.LSTM(lstm_units, return_sequences=True, input_shape=input_shape,
                   kernel_regularizer=l2(0.001)),
        layers.Dropout(dropout_rate),
        layers.LSTM(lstm_units//2, return_sequences=False),
        layers.Dropout(dropout_rate),
        layers.Dense(64, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='categorical_crossentropy',
        metrics=['accuracy',
                tf.keras.metrics.Precision(name='precision', top_k=1),
                tf.keras.metrics.Recall(name='recall', top_k=1)]
    )
    
    return model

print("Arquitecturas LSTM definidas:")
print("1. Binary LSTM:", create_lstm_binary((W, F)).summary())
print("\n2. Multiclass LSTM:", create_lstm_multiclass((W, F), num_classes).summary())

In [ ]:
class BalancedMetricsCallback(callbacks.Callback):
    def __init__(self, validation_data, patience=10):
        super().__init__()
        self.X_val, self.y_val = validation_data
        self.patience = patience
        self.best_f1 = 0
        self.best_weights = None
        self.wait = 0
        
    def on_epoch_end(self, epoch, logs=None):
        y_pred = self.model.predict(self.X_val, verbose=0)
        
        if self.model.output_shape[-1] == 1:  # Binario
            y_pred_class = (y_pred > 0.5).astype(int)
            precision = precision_score(self.y_val, y_pred_class, zero_division=0)
            recall = recall_score(self.y_val, y_pred_class, zero_division=0)
            
            if recall > 0.98:  # Penalizar recall demasiado alto
                adjusted_recall = 0.98
            else:
                adjusted_recall = recall
                
            if precision > 0 and adjusted_recall > 0:
                current_f1 = 2 * (precision * adjusted_recall) / (precision + adjusted_recall)
            else:
                current_f1 = 0
                
            if precision < 0.7:
                current_f1 = current_f1 * 0.8  # Penalizar baja precisión
                
        else:  # Multiclase
            y_pred_class = np.argmax(y_pred, axis=1)
            y_true_class = np.argmax(self.y_val, axis=1) if len(self.y_val.shape) > 1 else self.y_val
            precision = precision_score(y_true_class, y_pred_class, average='macro', zero_division=0)
            recall = recall_score(y_true_class, y_pred_class, average='macro', zero_division=0)
            current_f1 = f1_score(y_true_class, y_pred_class, average='macro', zero_division=0)
        
        # Guardar mejor modelo
        if current_f1 > self.best_f1:
            self.best_f1 = current_f1
            self.best_weights = self.model.get_weights()
            self.wait = 0
        else:
            self.wait += 1
            
        # Early stopping
        if self.wait >= self.patience:
            self.model.stop_training = True
            self.model.set_weights(self.best_weights)
            print(f"\nEarly stopping en época {epoch+1}. Restaurando mejores pesos.")
            
        print(f" - Val Precision: {precision:.4f}, Val Recall: {recall:.4f}, Val F1: {current_f1:.4f}")

In [ ]:
def objective_binary_lstm(trial):
    lstm_units = trial.suggest_categorical("lstm_units", [32, 64, 128])
    dropout_rate = trial.suggest_float("dropout_rate", 0.1, 0.5, step=0.1)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    
    use_second_lstm = trial.suggest_categorical("use_second_lstm", [True, False])
    dense_units = trial.suggest_categorical("dense_units", [16, 32, 64])
    
    model = models.Sequential()
    model.add(layers.LSTM(lstm_units, return_sequences=use_second_lstm, 
                         input_shape=(W, F), kernel_regularizer=l2(0.001)))
    model.add(layers.Dropout(dropout_rate))
    
    if use_second_lstm:
        model.add(layers.LSTM(lstm_units//2, return_sequences=False))
        model.add(layers.Dropout(dropout_rate/2))
    
    model.add(layers.Dense(dense_units, activation='relu'))
    model.add(layers.Dense(1, activation='sigmoid'))
    
    # Compilar
    model.compile(
        optimizer=Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    balanced_callback = BalancedMetricsCallback((X_val, y_bin_val), patience=7)
    
    history = model.fit(
        X_train, y_bin_train,
        validation_data=(X_val, y_bin_val),
        epochs=30,
        batch_size=batch_size,
        callbacks=[balanced_callback],
        verbose=0
    )
    
    y_pred = model.predict(X_val, verbose=0)
    y_pred_class = (y_pred > 0.5).astype(int)
    
    precision = precision_score(y_bin_val, y_pred_class, zero_division=0)
    recall = recall_score(y_bin_val, y_pred_class, zero_division=0)
    
    if recall > 0.98:
        adjusted_recall = 0.98
    else:
        adjusted_recall = recall
    
    if precision > 0 and adjusted_recall > 0:
        f1 = 2 * (precision * adjusted_recall) / (precision + adjusted_recall)
    else:
        f1 = 0
    
    if precision < 0.7:
        f1 = f1 * 0.8
    
    trial.set_user_attr("precision", float(precision))
    trial.set_user_attr("recall", float(recall))
    trial.set_user_attr("true_f1", float(f1_score(y_bin_val, y_pred_class, zero_division=0)))
    
    return f1

def objective_multiclass_lstm(trial):
    lstm_units = trial.suggest_categorical("lstm_units", [64, 128, 256])
    dropout_rate = trial.suggest_float("dropout_rate", 0.2, 0.5, step=0.1)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    
    model = create_lstm_multiclass((W, F), num_classes, lstm_units, dropout_rate, learning_rate)
    
    balanced_callback = BalancedMetricsCallback((X_val, y_multi_val), patience=7)
    
    history = model.fit(
        X_train, y_multi_train,
        validation_data=(X_val, y_multi_val),
        epochs=40,
        batch_size=batch_size,
        callbacks=[balanced_callback],
        verbose=0
    )
    
    y_pred = model.predict(X_val, verbose=0)
    y_pred_class = np.argmax(y_pred, axis=1)
    y_true_class = np.argmax(y_multi_val, axis=1)
    
    f1_macro = f1_score(y_true_class, y_pred_class, average='macro', zero_division=0)
    
    precision = precision_score(y_true_class, y_pred_class, average='macro', zero_division=0)
    recall = recall_score(y_true_class, y_pred_class, average='macro', zero_division=0)
    
    trial.set_user_attr("precision", float(precision))
    trial.set_user_attr("recall", float(recall))
    
    return f1_macro

In [ ]:
study_binary = optuna.create_study(
    direction="maximize",
    study_name="lstm_binary_anomaly_detection",
    sampler=optuna.samplers.TPESampler(seed=SEED)
)

study_binary.optimize(objective_binary_lstm, n_trials=20, show_progress_bar=True)

print("Best parameters:")
print(f"Mejor valor F1: {study_binary.best_value:.4f}")
print(f"Mejores parámetros: {study_binary.best_params}")
print(f"Precisión asociada: {study_binary.best_trial.user_attrs['precision']:.4f}")
print(f"Recall asociado: {study_binary.best_trial.user_attrs['recall']:.4f}")
print(f"F1 real (sin ajustes): {study_binary.best_trial.user_attrs['true_f1']:.4f}")

try:
    fig = optuna.visualization.plot_optimization_history(study_binary)
    fig.show()
except:
    print("No se pudo mostrar gráfico de optimización")

In [ ]:
best_params = study_binary.best_params

model_binary_final = models.Sequential()

model_binary_final.add(
    layers.LSTM(
        best_params['lstm_units'], 
        return_sequences=best_params['use_second_lstm'],
        input_shape=(W, F),
        kernel_regularizer=l2(0.001)
    )
)
model_binary_final.add(layers.Dropout(best_params['dropout_rate']))

if best_params['use_second_lstm']:
    model_binary_final.add(layers.LSTM(best_params['lstm_units']//2, return_sequences=False))
    model_binary_final.add(layers.Dropout(best_params['dropout_rate']/2))

model_binary_final.add(layers.Dense(best_params['dense_units'], activation='relu'))
model_binary_final.add(layers.Dense(1, activation='sigmoid'))

model_binary_final.compile(
    optimizer=Adam(learning_rate=best_params['learning_rate']),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)

print("Arquitectura del modelo final:")
model_binary_final.summary()

balanced_callback = BalancedMetricsCallback((X_val, y_bin_val), patience=10)
early_stopping = callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
reduce_lr = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

print("\nEntrenando modelo final...")
history_binary = model_binary_final.fit(
    X_train, y_bin_train,
    validation_data=(X_val, y_bin_val),
    epochs=50,
    batch_size=best_params['batch_size'],
    callbacks=[balanced_callback, early_stopping, reduce_lr],
    verbose=1
)

model_binary_final.save('lstm_binary_model.h5')
print("Modelo binario guardado como 'lstm_binary_model.h5'")

In [ ]:
test_loss, test_acc, test_precision, test_recall, test_auc = model_binary_final.evaluate(
    X_test, y_bin_test, verbose=0
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall: {test_recall:.4f}")
print(f"Test AUC: {test_auc:.4f}")

y_pred_proba = model_binary_final.predict(X_test, verbose=0)
y_pred_binary = (y_pred_proba > 0.5).astype(int)

test_f1 = f1_score(y_bin_test, y_pred_binary)
print(f"Test F1 Score: {test_f1:.4f}")

print("REPORT (Test Set)")
print(classification_report(y_bin_test, y_pred_binary, 
                           target_names=['Normal', 'Anomaly'],
                           digits=4))

# Matriz de confusión
cm = confusion_matrix(y_bin_test, y_pred_binary)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Normal', 'Anomaly'],
            yticklabels=['Normal', 'Anomaly'])
plt.title('Matriz de Confusión - LSTM Binario')
plt.ylabel('Verdadero')
plt.xlabel('Predicho')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

metrics = ['loss', 'accuracy', 'precision', 'recall']
titles = ['Pérdida', 'Exactitud', 'Precisión', 'Recall']

for i, metric in enumerate(metrics):
    axes[i].plot(history_binary.history[metric], label=f'Entrenamiento {metric}')
    axes[i].plot(history_binary.history[f'val_{metric}'], label=f'Validación {metric}')
    axes[i].set_title(f'Curva de {titles[i]}')
    axes[i].set_xlabel('Época')
    axes[i].set_ylabel(titles[i])
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
if num_classes > 2:
    study_multi = optuna.create_study(
        direction="maximize",
        study_name="lstm_multiclass_anomaly_detection",
        sampler=optuna.samplers.TPESampler(seed=SEED)
    )
    
    print(f"Iniciando optimización para {num_classes} clases...")
    study_multi.optimize(objective_multiclass_lstm, n_trials=15, show_progress_bar=True)
    
    print("\n" + "=" * 50)
    print("MEJORES HIPERPARÁMETROS ENCONTRADOS (Multiclase):")
    print("=" * 50)
    print(f"Mejor F1 Macro: {study_multi.best_value:.4f}")
    print(f"Mejores parámetros: {study_multi.best_params}")
    print(f"Precisión asociada: {study_multi.best_trial.user_attrs['precision']:.4f}")
    print(f"Recall asociado: {study_multi.best_trial.user_attrs['recall']:.4f}")
else:
    print("Solo hay 2 clases. Saltando optimización multiclase.")

In [ ]:
    best_params_multi = study_multi.best_params
    
    # Crear modelo multiclase final
    model_multi_final = create_lstm_multiclass(
        (W, F), 
        num_classes,
        lstm_units=best_params_multi['lstm_units'],
        dropout_rate=best_params_multi['dropout_rate'],
        learning_rate=best_params_multi['learning_rate']
    )
    
    print("Arquitectura del modelo multiclase final:")
    model_multi_final.summary()
    
    # Callbacks
    balanced_callback_multi = BalancedMetricsCallback((X_val, y_multi_val), patience=10)
    early_stopping_multi = callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
    reduce_lr_multi = callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
    
    # Entrenar
    print("\nEntrenando modelo multiclase final...")
    history_multi = model_multi_final.fit(
        X_train, y_multi_train,
        validation_data=(X_val, y_multi_val),
        epochs=60,
        batch_size=best_params_multi['batch_size'],
        callbacks=[balanced_callback_multi, early_stopping_multi, reduce_lr_multi],
        verbose=1
    )
    
    # Guardar modelo
    model_multi_final.save('lstm_multiclass_model.h5')
    print("Modelo multiclase guardado como 'lstm_multiclass_model.h5'")

In [ ]:
 
    # Evaluar en test set
    test_results_multi = model_multi_final.evaluate(X_test, y_multi_test, verbose=0)
    
    print(f"Test Loss: {test_results_multi[0]:.4f}")
    print(f"Test Accuracy: {test_results_multi[1]:.4f}")
    print(f"Test Precision: {test_results_multi[2]:.4f}")
    print(f"Test Recall: {test_results_multi[3]:.4f}")
    
    # Predicciones
    y_pred_multi_proba = model_multi_final.predict(X_test, verbose=0)
    y_pred_multi_class = np.argmax(y_pred_multi_proba, axis=1)
    y_true_multi_class = np.argmax(y_multi_test, axis=1)
    
    # Calcular F1 macro
    test_f1_macro = f1_score(y_true_multi_class, y_pred_multi_class, average='macro', zero_division=0)
    print(f"Test F1 Macro: {test_f1_macro:.4f}")
    
    # Reporte de clasificación detallado
    print("\n" + "=" * 50)
    print("REPORTE DE CLASIFICACIÓN (Test Set - Multiclase)")
    print("=" * 50)
    
    # Generar nombres de clases
    class_names = [f'Clase_{i}' for i in range(num_classes)]
    # Si la clase 0 es normal
    if 0 in y_true_multi_class:
        class_names[0] = 'Normal'
    
    print(classification_report(y_true_multi_class, y_pred_multi_class,
                               target_names=class_names,
                               digits=4))
    
    # Matriz de confusión (limitada si hay muchas clases)
    if num_classes <= 20:  # Mostrar si no hay demasiadas clases
        cm_multi = confusion_matrix(y_true_multi_class, y_pred_multi_class)
        plt.figure(figsize=(10, 8))
        sns.heatmap(cm_multi, annot=True, fmt='d', cmap='Blues',
                   xticklabels=class_names,
                   yticklabels=class_names)
        plt.title('Matriz de Confusión - LSTM Multiclase')
        plt.ylabel('Verdadero')
        plt.xlabel('Predicho')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()
    
    # Curva de aprendizaje multiclase
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    axes = axes.flatten()
    
    metrics_multi = ['loss', 'accuracy', 'precision', 'recall']
    titles_multi = ['Pérdida', 'Exactitud', 'Precisión', 'Recall']
    
    for i, metric in enumerate(metrics_multi):
        axes[i].plot(history_multi.history[metric], label=f'Entrenamiento {metric}')
        axes[i].plot(history_multi.history[f'val_{metric}'], label=f'Validación {metric}')
        axes[i].set_title(f'Curva de {titles_multi[i]} (Multiclase)')
        axes[i].set_xlabel('Época')
        axes[i].set_ylabel(titles_multi[i])
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()